<div style="background: linear-gradient(135deg, #1f3b57 0%, #2f6f8f 100%); color: white; padding: 28px 32px; border-radius: 10px;">
<h1 style="margin: 0; color: white;">t-SNE from Scratch</h1>
<p style="margin: 8px 0 0 0; font-size: 1.1em; opacity: 0.9;">Deriving and implementing t-Distributed Stochastic Neighbor Embedding, using only NumPy</p>
<p style="margin: 14px 0 0 0; font-size: 0.9em; opacity: 0.75;">Carles Trullàs</p>
</div>

This notebook builds **t-SNE** (t-Distributed Stochastic Neighbor Embedding) from first principles in NumPy and benchmarks it against `sklearn.manifold.TSNE`.

t-SNE is a non-linear dimensionality reduction method designed for visualisation: it maps high-dimensional points to 2 or 3 dimensions so that points that are neighbours in the original space stay neighbours in the embedding. Every piece of the algorithm (the perplexity calibration, the joint probabilities in both spaces, the KL objective, its gradient and the optimiser) is written explicitly so the mathematics maps one-to-one onto the code.

### Contents

1. [How t-SNE works](#1.-How-t-SNE-works)
2. [Setup](#2.-Setup)
3. [High-dimensional affinities](#3.-High-dimensional-affinities)
4. [Low-dimensional affinities, cost and gradient](#4.-Low-dimensional-affinities,-cost-and-gradient)
5. [The `TSNE` class](#5.-The-TSNE-class)
6. [Experiment 1: synthetic clusters](#6.-Experiment-1:-synthetic-clusters)
7. [Experiment 2: comparison with scikit-learn on digits](#7.-Experiment-2:-comparison-with-scikit-learn-on-digits)
8. [The effect of perplexity](#8.-The-effect-of-perplexity)
9. [Takeaways](#9.-Takeaways)

## 1. How t-SNE works

t-SNE turns distances into probabilities in both spaces and then moves the low-dimensional points until the two probability distributions match.

### Similarities in the original space

For each point $x_i$, the similarity to every other point $x_j$ is the conditional probability that $x_i$ would pick $x_j$ as its neighbour under a Gaussian centred at $x_i$:

$$
p_{j|i} = \frac{\exp\left(-\lVert x_i - x_j \rVert^2 / 2\sigma_i^2\right)}{\sum_{k \neq i} \exp\left(-\lVert x_i - x_k \rVert^2 / 2\sigma_i^2\right)}, \qquad p_{i|i} = 0.
$$

Each point gets its own bandwidth $\sigma_i$, so that dense and sparse regions are treated fairly. It is chosen so that the distribution $P_i$ has a fixed **perplexity**, a smooth measure of the effective number of neighbours:

$$
\text{Perp}(P_i) = 2^{H(P_i)}, \qquad H(P_i) = -\sum_{j} p_{j|i} \log_2 p_{j|i}.
$$

Perplexity grows monotonically with $\sigma_i$, so the right $\sigma_i$ can be found with a **binary search**. The conditional probabilities are then symmetrised into a joint distribution over all pairs:

$$
p_{ij} = \frac{p_{j|i} + p_{i|j}}{2n}.
$$

### Similarities in the embedding

In the low-dimensional space, similarities between $y_i$ and $y_j$ use a **Student-t distribution with one degree of freedom** (a Cauchy kernel), normalised over all pairs:

$$
q_{ij} = \frac{\left(1 + \lVert y_i - y_j \rVert^2\right)^{-1}}{\sum_{k \neq l} \left(1 + \lVert y_k - y_l \rVert^2\right)^{-1}}, \qquad q_{ii} = 0.
$$

The heavy tails are the key idea of t-SNE. A moderate distance in the original space can be represented by a much larger distance in the embedding, which relieves the *crowding problem* (there is not enough room in 2D to place all moderately distant points faithfully) and produces well-separated clusters.

### Objective and gradient

The embedding minimises the Kullback-Leibler divergence between $P$ and $Q$:

$$
C = \mathrm{KL}(P \,\|\, Q) = \sum_{i \neq j} p_{ij} \log \frac{p_{ij}}{q_{ij}},
$$

whose gradient has a remarkably simple form:

$$
\frac{\partial C}{\partial y_i} = 4 \sum_{j} \left(p_{ij} - q_{ij}\right) \left(y_i - y_j\right) \left(1 + \lVert y_i - y_j \rVert^2\right)^{-1}.
$$

It reads as a system of springs: pairs with $p_{ij} > q_{ij}$ (too far apart in the embedding) attract each other and pairs with $p_{ij} < q_{ij}$ repel.

### Optimisation

The embedding starts as a tiny random (or PCA) cloud and is updated by gradient descent with momentum:

$$
\mathcal{Y}^{(t)} = \mathcal{Y}^{(t-1)} - \eta \, \frac{\partial C}{\partial \mathcal{Y}} + \alpha(t) \left(\mathcal{Y}^{(t-1)} - \mathcal{Y}^{(t-2)}\right).
$$

Two tricks make it work well in practice:

- **Early exaggeration.** During the first iterations every $p_{ij}$ is multiplied by a factor (typically 12). This makes attractive forces dominate, so clusters form tight, well-separated groups early on and have room to move around.
- **Adaptive gains.** Each coordinate has its own step multiplier, which grows when the gradient keeps its sign and shrinks when it flips.

## 2. Setup

Only NumPy is needed for the algorithm itself. Scikit-learn is used for the datasets, as a reference implementation and to compute the trustworthiness metric.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits, make_blobs
from sklearn.manifold import TSNE as SklearnTSNE
from sklearn.manifold import trustworthiness
from sklearn.preprocessing import StandardScaler

plt.rcParams.update({
    "figure.figsize": (10, 4.5),
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

CUSTOM_COLOR = "#2f6f8f"
NATIVE_COLOR = "#e07a3f"
CMAP = "tab10"
SEED = 42

## 3. High-dimensional affinities

### Pairwise squared distances

Everything starts from the matrix of squared Euclidean distances. Using $\lVert a - b \rVert^2 = \lVert a \rVert^2 + \lVert b \rVert^2 - 2\,a^\top b$ it can be computed with a single matrix product instead of a double loop. The same function is reused for the embedding.

In [ ]:
def pairwise_squared_distances(X):
    # Squared Euclidean distance between every pair of rows of X.
    sq_norms = np.sum(X ** 2, axis=1)
    D = sq_norms[:, None] + sq_norms[None, :] - 2.0 * X @ X.T
    np.maximum(D, 0.0, out=D)  # remove tiny negatives caused by rounding
    np.fill_diagonal(D, 0.0)
    return D

### Perplexity calibration

For a single point, `row_entropy` evaluates the conditional distribution $P_i$ for a given precision $\beta_i = 1 / (2\sigma_i^2)$ and returns it together with its entropy. Working with $\beta$ and natural logarithms is equivalent to the formulas above (the target becomes $\log(\text{perplexity})$ instead of $\log_2$), and subtracting the largest logit before exponentiating keeps the computation numerically stable.

`conditional_probabilities` then runs a binary search on $\beta_i$ for every point:

- if the entropy is **too high**, the Gaussian is too wide, so $\beta_i$ must increase;
- if it is **too low**, the Gaussian is too narrow, so $\beta_i$ must decrease.

While no upper (or lower) bound has been found yet, $\beta_i$ is doubled (or halved); afterwards the search bisects the interval. A few dozen steps are enough to reach the target perplexity to high precision.

In [ ]:
def row_entropy(distances, beta):
    # Conditional probabilities p_{j|i} and their entropy (in nats) for one point.
    logits = -distances * beta
    logits -= logits.max()
    p = np.exp(logits)
    p /= p.sum()
    entropy = -np.sum(p * np.log(np.maximum(p, 1e-12)))
    return p, entropy


def conditional_probabilities(D, perplexity, tol=1e-5, max_iter=100):
    # Binary search for each sigma_i so that Perp(P_i) matches the target perplexity.
    n = D.shape[0]
    target_entropy = np.log(perplexity)
    P = np.zeros((n, n))
    betas = np.ones(n)

    for i in range(n):
        others = np.arange(n) != i
        distances = D[i, others]

        beta, beta_min, beta_max = 1.0, 0.0, np.inf
        for _ in range(max_iter):
            p, entropy = row_entropy(distances, beta)
            diff = entropy - target_entropy
            if abs(diff) < tol:
                break
            if diff > 0:  # too flat: narrow the Gaussian
                beta_min = beta
                beta = beta * 2.0 if beta_max == np.inf else (beta + beta_max) / 2.0
            else:         # too peaked: widen the Gaussian
                beta_max = beta
                beta = beta / 2.0 if beta_min == 0.0 else (beta + beta_min) / 2.0

        P[i, others] = p
        betas[i] = beta

    sigmas = np.sqrt(1.0 / (2.0 * betas))
    return P, sigmas

### Joint probabilities

Symmetrising guarantees that every point, even an outlier whose conditional probabilities towards it are all tiny, contributes at least $1/(2n)$ to the total mass, so it still has a meaningful pull on the embedding.

In [ ]:
def joint_probabilities(X, perplexity):
    # Symmetric joint distribution P over all pairs of points.
    n = X.shape[0]
    D = pairwise_squared_distances(X)
    P_cond, sigmas = conditional_probabilities(D, perplexity)
    P = (P_cond + P_cond.T) / (2.0 * n)
    P = np.maximum(P, 1e-12)
    np.fill_diagonal(P, 0.0)
    return P, P_cond, sigmas

## 4. Low-dimensional affinities, cost and gradient

`low_dim_affinities` computes the Student-t kernel and the normalised $Q$. The unnormalised kernel `num` is returned as well, because the gradient needs it.

The gradient is fully vectorised. Writing $W_{ij} = (p_{ij} - q_{ij})(1 + \lVert y_i - y_j \rVert^2)^{-1}$, the sum over $j$ becomes

$$
\frac{\partial C}{\partial y_i} = 4 \left( y_i \sum_j W_{ij} - \sum_j W_{ij} \, y_j \right),
$$

which in matrix form is $4\,(\mathrm{diag}(W\mathbf{1})\,Y - W Y)$: one row sum and one matrix product for all points at once.

In [ ]:
def low_dim_affinities(Y):
    # Student-t kernel and normalised joint distribution Q in the embedding.
    num = 1.0 / (1.0 + pairwise_squared_distances(Y))
    np.fill_diagonal(num, 0.0)
    Q = np.maximum(num / num.sum(), 1e-12)
    np.fill_diagonal(Q, 0.0)
    return Q, num


def kl_divergence(P, Q):
    # KL(P || Q), ignoring the diagonal.
    mask = P > 0
    return float(np.sum(P[mask] * np.log(P[mask] / Q[mask])))


def kl_gradient(P, Q, num, Y):
    # Gradient of KL(P || Q) with respect to the embedding Y.
    W = (P - Q) * num
    return 4.0 * (W.sum(axis=1)[:, None] * Y - W @ Y)

### Gradient check

Before training anything, it is worth verifying the analytical gradient against central finite differences on a small random problem. The relative error should be several orders of magnitude below 1.

In [ ]:
rng = np.random.default_rng(0)
X_small = rng.normal(size=(20, 5))
Y_small = rng.normal(size=(20, 2))
P_small, _, _ = joint_probabilities(X_small, perplexity=5)

Q_small, num_small = low_dim_affinities(Y_small)
analytic = kl_gradient(P_small, Q_small, num_small, Y_small)

numeric = np.zeros_like(Y_small)
eps = 1e-6
for i in range(Y_small.shape[0]):
    for d in range(Y_small.shape[1]):
        Y_plus, Y_minus = Y_small.copy(), Y_small.copy()
        Y_plus[i, d] += eps
        Y_minus[i, d] -= eps
        numeric[i, d] = (kl_divergence(P_small, low_dim_affinities(Y_plus)[0])
                         - kl_divergence(P_small, low_dim_affinities(Y_minus)[0])) / (2 * eps)

rel_error = np.linalg.norm(analytic - numeric) / np.linalg.norm(analytic + numeric)
print(f"Relative error between analytical and numerical gradient: {rel_error:.2e}")

## 5. The `TSNE` class

The class wraps the pieces above behind a scikit-learn style `fit_transform`. The training loop:

1. Compute $P$ once from the data, with the requested perplexity.
2. Initialise $Y$ either with small Gaussian noise or with the first principal components rescaled to a tiny standard deviation. PCA initialisation preserves more of the global structure and makes runs reproducible.
3. For every iteration, compute $Q$, the gradient and the KL divergence, and update $Y$ with momentum and adaptive gains.
4. During the first `early_exaggeration_iter` iterations, use $P$ multiplied by `early_exaggeration` and a lower momentum (0.5); afterwards switch to the true $P$ and a higher momentum (0.8).

With `learning_rate="auto"` the step size is set to $\max(n / (4 \cdot \text{early exaggeration}),\, 50)$, the heuristic that scikit-learn also uses. The embedding is re-centred after each step, since the cost is invariant to translations.

The KL divergence is recorded at every iteration (always against the true $P$, so the curve is comparable across phases) to inspect convergence afterwards.

In [ ]:
class TSNE:
    # Exact t-SNE implemented with NumPy.

    def __init__(self, n_components=2, perplexity=30.0, learning_rate="auto",
                 max_iter=1000, early_exaggeration=12.0, early_exaggeration_iter=250,
                 initial_momentum=0.5, final_momentum=0.8, min_gain=0.01,
                 init="pca", random_state=None, verbose=0):
        self.n_components = n_components
        self.perplexity = perplexity
        self.learning_rate = learning_rate
        self.max_iter = max_iter
        self.early_exaggeration = early_exaggeration
        self.early_exaggeration_iter = early_exaggeration_iter
        self.initial_momentum = initial_momentum
        self.final_momentum = final_momentum
        self.min_gain = min_gain
        self.init = init
        self.random_state = random_state
        self.verbose = verbose

    # ------------------------------------------------------------------
    def _initial_embedding(self, X, rng):
        n = X.shape[0]
        if self.init == "pca":
            Xc = X - X.mean(axis=0)
            U, S, _ = np.linalg.svd(Xc, full_matrices=False)
            Y = U[:, :self.n_components] * S[:self.n_components]
            return Y / np.std(Y[:, 0]) * 1e-4
        if self.init == "random":
            return rng.normal(scale=1e-4, size=(n, self.n_components))
        raise ValueError("init must be 'pca' or 'random'")

    def fit_transform(self, X):
        X = np.asarray(X, dtype=float)
        n = X.shape[0]
        if not 1 < self.perplexity < n:
            raise ValueError("perplexity must be between 1 and the number of samples")
        rng = np.random.default_rng(self.random_state)

        lr = (max(n / self.early_exaggeration / 4.0, 50.0)
              if self.learning_rate == "auto" else float(self.learning_rate))

        P, self.conditional_P_, self.sigmas_ = joint_probabilities(X, self.perplexity)
        Y = self._initial_embedding(X, rng)
        update = np.zeros_like(Y)
        gains = np.ones_like(Y)
        self.kl_history_ = []

        for it in range(self.max_iter):
            exaggerating = it < self.early_exaggeration_iter
            P_eff = P * self.early_exaggeration if exaggerating else P
            momentum = self.initial_momentum if exaggerating else self.final_momentum

            Q, num = low_dim_affinities(Y)
            grad = kl_gradient(P_eff, Q, num, Y)
            self.kl_history_.append(kl_divergence(P, Q))

            # Adaptive gains: grow when the direction is stable, shrink when it flips
            same_sign = np.sign(grad) == np.sign(update)
            gains = np.where(same_sign, gains * 0.8, gains + 0.2)
            np.maximum(gains, self.min_gain, out=gains)

            update = momentum * update - lr * gains * grad
            Y = Y + update
            Y -= Y.mean(axis=0)

            if self.verbose and (it + 1) % self.verbose == 0:
                phase = "exaggeration" if exaggerating else "optimisation"
                print(f"Iteration {it + 1:>5}/{self.max_iter} ({phase:<12}) "
                      f"KL = {self.kl_history_[-1]:.4f}")

        self.learning_rate_ = lr
        self.embedding_ = Y
        self.kl_divergence_ = kl_divergence(P, low_dim_affinities(Y)[0])
        self.n_iter_ = self.max_iter
        return Y

### Plotting helper

Draws an embedding coloured by the true labels. The labels are **never** used by t-SNE; they only help to judge whether the structure found is meaningful.

In [ ]:
def plot_embedding(ax, Y, labels, title):
    scatter = ax.scatter(Y[:, 0], Y[:, 1], c=labels, cmap=CMAP, s=8, alpha=0.85)
    ax.set(title=title, xticks=[], yticks=[])
    ax.grid(False)
    return scatter

## 6. Experiment 1: synthetic clusters

A synthetic dataset with 1,000 points in 10 dimensions drawn from 10 Gaussian clusters with different spreads. In 10 dimensions there is no way to look at the clusters directly; a good embedding should reveal all ten.

In [ ]:
X_blobs, y_blobs = make_blobs(
    n_samples=1000, n_features=10, centers=10,
    cluster_std=np.linspace(1.0, 3.0, 10), random_state=SEED,
)
X_blobs = StandardScaler().fit_transform(X_blobs)

print(f"Data: {X_blobs.shape}  |  Points per cluster: {np.bincount(y_blobs).tolist()}")

### Checking the perplexity calibration

Before running the optimisation, it is worth confirming that the binary search does its job: the perplexity of every conditional distribution $P_i$ should match the target, while the bandwidths $\sigma_i$ adapt to the local density (points in the wider clusters need larger $\sigma_i$).

In [ ]:
PERPLEXITY = 30
_, P_cond_blobs, sigmas_blobs = joint_probabilities(X_blobs, PERPLEXITY)

row_entropies = -np.sum(P_cond_blobs * np.log(np.maximum(P_cond_blobs, 1e-12)), axis=1)
achieved_perplexity = np.exp(row_entropies)
print(f"Target perplexity: {PERPLEXITY}")
print(f"Achieved perplexity: min = {achieved_perplexity.min():.4f}, "
      f"max = {achieved_perplexity.max():.4f}")

fig, ax = plt.subplots(figsize=(9, 4))
sigma_by_cluster = [sigmas_blobs[y_blobs == k] for k in range(10)]
ax.boxplot(sigma_by_cluster, showfliers=False)
ax.set(xlabel="Cluster (ordered by increasing spread)", ylabel=r"$\sigma_i$",
       title="Calibrated bandwidths adapt to the local density",
       xticks=range(1, 11), xticklabels=range(10))
plt.show()

### Training

1,000 iterations, of which the first 250 use early exaggeration.

In [ ]:
tsne_blobs = TSNE(perplexity=PERPLEXITY, random_state=SEED, verbose=100)

start = time.perf_counter()
Y_blobs = tsne_blobs.fit_transform(X_blobs)
print(f"\nTraining time: {time.perf_counter() - start:.1f} s  |  "
      f"learning rate: {tsne_blobs.learning_rate_:.1f}  |  "
      f"final KL: {tsne_blobs.kl_divergence_:.4f}")

On the left, the final embedding coloured by the (unused) cluster labels. On the right, the KL divergence during training. It falls quickly while the clusters form, flattens towards the end of early exaggeration (when the optimiser is fitting the exaggerated $P$, not the true one), and drops again as soon as the true $P$ is used.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

plot_embedding(ax1, Y_blobs, y_blobs, "Custom t-SNE on 10-D synthetic clusters")

ax2.plot(tsne_blobs.kl_history_, color=CUSTOM_COLOR)
ax2.axvline(tsne_blobs.early_exaggeration_iter, color="grey", ls="--", lw=1,
            label="End of early exaggeration")
ax2.set(xlabel="Iteration", ylabel="KL(P || Q)", title="Convergence")
ax2.legend()

plt.tight_layout()
plt.show()

### How the embedding evolves

Snapshots of the same run at several iterations show the two phases of the optimisation: during early exaggeration the clusters condense into tight blobs, and once it ends they expand and settle into their final layout.

In [ ]:
snapshots = [50, 250, 400, 1000]
fig, axes = plt.subplots(1, len(snapshots), figsize=(16, 4))

for ax, n_iter in zip(axes, snapshots):
    model = TSNE(perplexity=PERPLEXITY, max_iter=n_iter, random_state=SEED)
    plot_embedding(ax, model.fit_transform(X_blobs), y_blobs, f"Iteration {n_iter}")

plt.tight_layout()
plt.show()

## 7. Experiment 2: comparison with scikit-learn on digits

The digits dataset contains 1,797 handwritten digits as 8×8 images, i.e. 64-dimensional vectors. Both implementations are run with the same perplexity, learning rate, number of iterations, early exaggeration and PCA initialisation. Scikit-learn uses the Barnes-Hut approximation by default, which brings the cost of each iteration down from $O(n^2)$ to $O(n \log n)$.

Two metrics are used to compare them:

- **KL divergence**, the objective itself (lower is better). Each implementation reports its own value; scikit-learn's Barnes-Hut value is an approximation.
- **Trustworthiness**, which measures (between 0 and 1) to what extent the nearest neighbours of each point in the embedding are also close in the original space.

In [ ]:
X_digits, y_digits = load_digits(return_X_y=True)
print(f"Data: {X_digits.shape}  |  Classes: {np.unique(y_digits).tolist()}")

In [ ]:
common = dict(n_components=2, perplexity=30, learning_rate="auto", max_iter=1000,
              early_exaggeration=12.0, init="pca", random_state=SEED)

custom_tsne = TSNE(**common)
start = time.perf_counter()
Y_custom = custom_tsne.fit_transform(X_digits)
custom_time = time.perf_counter() - start

native_tsne = SklearnTSNE(**common)
start = time.perf_counter()
Y_native = native_tsne.fit_transform(X_digits)
native_time = time.perf_counter() - start

results = pd.DataFrame([
    {"Model": "Custom t-SNE (exact)", "Time (s)": round(custom_time, 2),
     "KL divergence": round(custom_tsne.kl_divergence_, 4),
     "Trustworthiness (k=10)": round(trustworthiness(X_digits, Y_custom, n_neighbors=10), 4)},
    {"Model": "scikit-learn (Barnes-Hut)", "Time (s)": round(native_time, 2),
     "KL divergence": round(native_tsne.kl_divergence_, 4),
     "Trustworthiness (k=10)": round(trustworthiness(X_digits, Y_native, n_neighbors=10), 4)},
])
results

### Visual comparison

Both embeddings should show the ten digit classes as separate groups, with the same small satellite groups of atypical digits (for example, some 1s and 9s written in an unusual way). The absolute positions, rotations and reflections of the clusters are arbitrary and can differ between implementations.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))

plot_embedding(ax1, Y_custom, y_digits, "Custom t-SNE")
scatter = plot_embedding(ax2, Y_native, y_digits, "scikit-learn TSNE")

for ax, Y in ((ax1, Y_custom), (ax2, Y_native)):
    for digit in range(10):
        cx, cy = np.median(Y[y_digits == digit], axis=0)
        ax.text(cx, cy, str(digit), fontsize=14, fontweight="bold", ha="center", va="center",
                bbox=dict(boxstyle="circle,pad=0.2", fc="white", ec="none", alpha=0.8))

fig.colorbar(scatter, ax=[ax1, ax2], ticks=range(10), label="Digit", shrink=0.8)
plt.show()

## 8. The effect of perplexity

Perplexity is the main hyperparameter of t-SNE: it sets how many neighbours each point "pays attention to". Small values focus on very local structure, giving looser clusters and, on real data, a tendency to break groups into fragments; large values take more of the global structure into account and produce tighter, more compact clusters. Typical values lie between 5 and 50, and it is good practice to try several.

In [ ]:
perplexities = [5, 30, 100]
fig, axes = plt.subplots(1, len(perplexities), figsize=(15, 4.8))

for ax, perp in zip(axes, perplexities):
    model = TSNE(perplexity=perp, random_state=SEED)
    Y = model.fit_transform(X_blobs)
    plot_embedding(ax, Y, y_blobs, f"Perplexity = {perp}  (KL = {model.kl_divergence_:.3f})")

plt.tight_layout()
plt.show()

Note that the KL divergences are not comparable across perplexities, since each one is measured against a different $P$.

## 9. Takeaways

- **t-SNE is probability matching.** Distances become neighbour probabilities in both spaces (Gaussian in the original space, Student-t in the embedding) and the embedding is moved to minimise $\mathrm{KL}(P \,\|\, Q)$.
- **Perplexity sets a per-point bandwidth.** A binary search finds a different $\sigma_i$ for every point so that each one has the same effective number of neighbours, which makes the method adapt to regions of different density.
- **The heavy-tailed kernel is what makes clusters pop out.** It lets moderately distant points be placed far apart in 2D, which solves the crowding problem of its predecessor SNE.
- **The gradient is a system of springs** and vectorises into one row sum and one matrix product. Early exaggeration, momentum and adaptive gains are what turn plain gradient descent into a reliable optimiser.
- **Read t-SNE plots with care.** The method preserves local neighbourhoods, but cluster sizes and distances between clusters are not meaningful, and results depend on perplexity and initialisation.
- **What scikit-learn adds on top** is mostly scalability: the Barnes-Hut approximation (and sparse nearest-neighbour affinities) reduces the cost from $O(n^2)$ to $O(n \log n)$ per iteration, which is what makes t-SNE usable on tens of thousands of points. The exact implementation here is fine for a few thousand.